# 00 · Quickstart

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/microsoft/SPARROW-Playground/blob/main/notebooks/00_quickstart.ipynb)

Detect animals in a folder of camera-trap images in four steps: install the engine,
get a model from the Sparrow Model Zoo, run detection, look at the result.

SPARROW-Playground runs on [Sparrow-Engine](https://github.com/microsoft/SPARROW-Engine),
an ONNX-based inference engine. There is no PyTorch dependency.

In [ ]:
# Setup: install sparrow-engine if it is missing (for example on Colab).
# For an NVIDIA GPU, install `sparrow-engine-gpu` instead (same Python API).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("sparrow_engine") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sparrow-engine[video]>=0.1.31"])

import sparrow_engine as spe

print("sparrow-engine", spe.__version__, "| device:", spe.active_device())

In [ ]:
# Demo data: downloaded once into ./data and checked against a published checksum.
import hashlib
import os
import shutil
import urllib.request
import zipfile
from pathlib import Path

DATA = Path(os.environ.get("SPP_DATA_DIR", "data")).resolve()
ASSETS = {
    # Pytorch-Wildlife demo assets, CC-BY-4.0, https://zenodo.org/records/15376499
    "demo_data_det.zip": (
        "https://zenodo.org/records/15376499/files/demo_data_det.zip?download=1",
        "md5", "373e15c8e7c8894379a82086a9177763",
    ),
    "demo_data_base.zip": (
        "https://zenodo.org/records/15376499/files/demo_data_base.zip?download=1",
        "md5", "f89b9b76c753d90e4296d62d71cf1732",
    ),
    # "Yellow-Throated Vireo Song" by SwampVids, CC-BY-4.0, Wikimedia Commons
    "vireo.wav": (
        "https://upload.wikimedia.org/wikipedia/commons/4/4f/"
        "Vireo_flavifrons_-_Yellow-Throated_Vireo_Song.wav",
        "sha1", "2b472209c1b18bfc0c225975838e3a34c1e9887a",
    ),
}


def fetch(name):
    """Download an asset (once), verify it, and unzip archives into DATA."""
    url, algo, digest = ASSETS[name]
    target = DATA / name
    DATA.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        print(f"downloading {name} ...")
        part = target.with_name(target.name + ".part")
        req = urllib.request.Request(url, headers={"User-Agent": "sparrow-playground"})
        with urllib.request.urlopen(req) as r, open(part, "wb") as f:
            shutil.copyfileobj(r, f)
        part.rename(target)
    h = hashlib.new(algo)
    with open(target, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    if h.hexdigest() != digest:
        target.unlink()
        raise RuntimeError(f"{name}: checksum mismatch, deleted; run the cell again")
    marker = DATA / f".{name}.extracted"
    if name.endswith(".zip") and not marker.exists():
        with zipfile.ZipFile(target) as z:
            z.extractall(DATA)
        marker.touch()
    return target

In [ ]:
fetch("demo_data_det.zip")
images = DATA / "data/data_example/images"
print(len(list(images.rglob("*.jpg"))), "images in", images)

In [ ]:
# Models are installed into SPARROW_ENGINE_MODEL_DIR (default ~/.sparrow-engine/models).
# Already-installed models are skipped. Inference never downloads anything by itself.
for model_id in ["MDV6-yolov10-c"]:
    spe.download_model(model_id)

Run MegaDetector V6 on every image in the folder. Each result carries the path it belongs to.

In [ ]:
results = spe.detect(images, model="MDV6-yolov10-c", threshold=0.3, recursive=True)
for r in results[:5]:
    print(Path(r.path).name, [(d.label, round(d.confidence, 2)) for d in r.detections])

In [ ]:
spe.summarize(results)

In [ ]:
from IPython.display import Image, display

png = spe.visualize(results[:1])[0]
display(Image(data=png, width=640))

## Next

| Notebook | Topic |
|---|---|
| 01 | Browse and download models from the zoo |
| 02 | Image detection in depth |
| 03 | Detection + species classification |
| 04 | Bioacoustics |
| 05 | Image embeddings and similarity search |
| 06 | Export to MegaDetector JSON, COCO, CSV |
| 07 | Day/night and visualization |
| 08 | Video |
| 09 | Webcam and RTSP streams |
| 10 | Coming from Pytorch-Wildlife |
| 11-15 | Receive field data with sparrow-comm (device, FTPS, Reconyx, e-mail) and process it |
| 16 | Sparrow Engine as an HTTP server (needs Docker) |